In [1]:
text = "Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."
tokens = text.encode("utf-8")
tokens = list(map(int, tokens))

print(text)
print(tokens)
len(tokens)

Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception.
[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 226, 128, 140, 240, 159, 135, 169

616

In [19]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
    newids= []
    i = 0
    while i < len(ids) :
        if i < len(ids ) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i+=1

    return newids

vocab_size = 276
num_merges = vocab_size - 256

ids = list(tokens) # creating a copy of the list

# print(ids)
merges =  {}
for i in range(num_merges):
    stats= get_stats(ids)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    print(f"merging pair {pair} into a new token {idx}")
    ids = merge(ids, pair, idx)
    merges[pair] = idx

print(merges)
print(ids)

merging pair (101, 32) into a new token 256
merging pair (240, 159) into a new token 257
merging pair (226, 128) into a new token 258
merging pair (105, 110) into a new token 259
merging pair (115, 32) into a new token 260
merging pair (97, 110) into a new token 261
merging pair (116, 104) into a new token 262
merging pair (257, 133) into a new token 263
merging pair (257, 135) into a new token 264
merging pair (97, 114) into a new token 265
merging pair (239, 189) into a new token 266
merging pair (258, 140) into a new token 267
merging pair (267, 264) into a new token 268
merging pair (101, 114) into a new token 269
merging pair (111, 114) into a new token 270
merging pair (116, 32) into a new token 271
merging pair (259, 103) into a new token 272
merging pair (115, 116) into a new token 273
merging pair (261, 100) into a new token 274
merging pair (32, 262) into a new token 275
{(101, 32): 256, (240, 159): 257, (226, 128): 258, (105, 110): 259, (115, 32): 260, (97, 110): 261, (116, 

In [3]:
print(f"compression ratio = {len(tokens) / len(ids):.2f}")

compression ratio = 1.37


In [18]:
vocab = {idx:bytes([idx]) for idx in range(256)}
for (p0,p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

# print(vocab)
def decode(ids):
    # for id in ids
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf-8", errors="replace")
    return text

decode([128])

'�'

In [27]:
def encode(text):
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key= lambda p : merges.get(p, float('inf')))
        if pair not in merges:
            break
        idx = merges[pair]

        tokens = merge(tokens, pair, idx)
    return tokens

decode(encode("H"))

'H'

In [28]:
text2 = decode(encode(text))
text == text2

True